In [11]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [13]:
# Load red wine data
red_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
red = pd.read_csv(red_url, sep=';')

# Load white wine data
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
white = pd.read_csv(white_url, sep=';')

print("Red wine shape:", red.shape)
print("White wine shape:", white.shape)

Red wine shape: (1599, 12)
White wine shape: (4898, 12)


In [14]:
# add wine type column
red['type'] = 0
white['type'] = 1

# combine datasets
wine = pd.concat([red, white], ignore_index=True)

wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


In [15]:
X = wine.drop("quality", axis=1)
y = wine["quality"]

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(5197, 12) (1300, 12)


In [16]:
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [17]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R² Score:", r2)

RMSE: 0.735689101706392
R² Score: 0.267157485126273


In [18]:
import joblib

joblib.dump(model, "wine_quality_model.pkl")
print("Model saved!")

Model saved!


In [19]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", ascending=False)

coefficients

,Feature,Coefficient
9,sulphates,0.763015
8,pH,0.527633
10,alcohol,0.232594
0,fixed acidity,0.092133
3,residual sugar,0.062822
5,free sulfur dioxide,0.005978
6,total sulfur dioxide,-0.001632
2,citric acid,-0.097888
11,type,-0.335208
4,chlorides,-0.582632


In [20]:
sample = X_test.iloc[0:1]
prediction = model.predict(sample)

print("Predicted Quality:", prediction[0])
print("Actual Quality:", y_test.iloc[0])

Predicted Quality: 6.693327284423347
Actual Quality: 8


In [21]:
wine.to_csv("wine_combined.csv", index=False)

In [2]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

role = sagemaker.get_execution_role()

estimator = SKLearn(
    entry_point="train_sagemaker.py",
    source_dir="/home/sagemaker-user/sm_job",
    role=role,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
)

estimator.fit()

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-23 02:21:12 Starting - Starting the training job...
2026-02-23 02:21:28 Starting - Preparing the instances for training...
2026-02-23 02:22:15 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-